# Load categorized clinical tables

The package splits measurement rows into labs, vision, physical assessment, MoCA and monofilament tables. Each stays in long format. Original fields and repeated measurements are preserved; the curated mapping adds `variable_name`.

In [ ]:
from pathlib import Path
from aireadi_preprocessing import load_clinical
from aireadi_preprocessing.tables import clinical_mappings

dataset = Path("/path/to/ai-readi")
output = Path("/path/to/processed")

## Inspect the mapping

The mapping uses measurement concept IDs to choose a category and readable variable name. Source labels can be truncated or difficult to search.

| Identifier | Category | Standardized name |
| --- | --- | --- |
| 3004410 | lab_test | HbA1c (%) |
| 3004501 | lab_test | Glucose (mg/dL) |
| 2005200344 | moca | MoCA Trail Making Score |


In [ ]:
mapping = clinical_mappings()
{k: mapping["lab_test"][k] for k in ["3004410", "3004501"]}

## Load the categories

`load_clinical` returns a dictionary of DataFrames. Observation rows are included if `observation.csv` is present. Unknown measurement codes get their own `unmapped_measurements` table.

In [ ]:
tables = load_clinical(dataset)
{name: len(frame) for name, frame in tables.items()}

In [ ]:
labs = tables["lab_test"]
labs.head()

## Select one measurement

Keep rows whose standardized name matches the measurement you want. Each participant can still have more than one row.

In [ ]:
hba1c = labs.loc[labs["variable_name"].eq("HbA1c (%)")]
hba1c.head()

## Save the category CSVs

Choose a new local folder outside the source dataset. Each file retains all original columns plus `variable_name`.

In [ ]:
output.mkdir(parents=True, exist_ok=False)
for category, frame in tables.items():
    frame.to_csv(output / f"{category}.csv", index=False)